# Tutorial: SQL analytics on the GEDI H3 database with DuckDB

The GEDI H3 dataset is laid out in a format such that many distributed frameworks will be able to help you quickly scan and query the data. These tools, which include DuckDB, Spark, Dask, and others, are designed to help you run your usual dataframe code over datasets that are too large to fit in memory. You can use these tools alone or in combination with geopandas/R dataframes -- for example, selecting a subset of the data from the database with a fast query framework and then converting it to your standard library for further processing.

Below is an introduction to using [DuckDB](https://duckdb.org/) to work with the GEDI data.


## 1. Introduction to DuckDB and SQL

DuckDB is a query engine that supports efficient querying of large datasets split among many files. It queries using SQL, a standard language for interacting with tabular data. The DuckDB SQL API docs are [here](https://duckdb.org/docs/stable/sql/introduction), and generally most internet advice for SQL (especially the dialect called "PostgreSQL") will also work in DuckDB.

DuckDB also can be used in R with the package duckplyr. All the SQL commands shown below will be the same in R, but instead of con.sql("SQL COMMAND") you will run dbGetQuery(con, "SQL COMMAND"). For more details, see the R client API docs [here](https://duckdb.org/docs/stable/clients/r).

We start out by creating a DuckDB connection and attaching to the GEDI database.

In [1]:
%load_ext autoreload
%autoreload 2
from gedih3 import sqlutils

con = sqlutils.init_duckdb()
# Reference the GEDI database in DuckDB as 'gedi_dl.data'
sqlutils.attach_ducklake_db(con, 'gedi_dl')

Loading environment variables from /home/ameliah/.gedih3.env


### SQL: SELECT and FROM

In [2]:
# The simplest SQL query has two clauses: SELECT and FROM
# SELECT specifies which columns to return
# FROM specifies which table to query

# Try modifying the list of columns below to fetch other data.
# Try a mistyped column name (e.g. "SELECT rh_98 FROM gedi_dl.data") 
#   to see how DuckDB reports errors.
# Note that column names with a '/' in them need to be double-quoted.
con.sql("""--sql
    SELECT 
        shot_number, 
        agbd_l4a, 
        rh_098_l2a, 
        l4_quality_flag_l4a, 
        "land_cover_data/leaf_off_flag_l2a",
        h3_03,
    FROM gedi_dl.data
""")

┌───────────────────┬──────────┬────────────┬─────────────────────┬───────────────────────────────────┬─────────────────┐
│    shot_number    │ agbd_l4a │ rh_098_l2a │ l4_quality_flag_l4a │ land_cover_data/leaf_off_flag_l2a │      h3_03      │
│      uint64       │  float   │   float    │        uint8        │               uint8               │     varchar     │
├───────────────────┼──────────┼────────────┼─────────────────────┼───────────────────────────────────┼─────────────────┤
│ 49120800100051876 │  -9999.0 │        0.0 │                   0 │                                 0 │ 838b5cfffffffff │
│ 49120800100051877 │  -9999.0 │        0.0 │                   0 │                                 0 │ 838b5cfffffffff │
│ 49120800100051878 │  -9999.0 │        0.0 │                   0 │                                 0 │ 838b5cfffffffff │
│ 49120800100051879 │  -9999.0 │        0.0 │                   0 │                                 0 │ 838b5cfffffffff │
│ 49120800100051880 │  -


A few things to notice about the above query:
- It's very fast (< 1 second). No individual parquet file was fully loaded into memory to produce the output.
- It's not showing all the data, just a preview of 20 rows. Not all parquet files were scanned to produce the output.
- If you restart the kernel and run the query again, you may get a different subset of rows in the preview.

You can select all columns using `*` and use the `.columns` function to get the complete list of columns in the database, e.g.:
```python
con.sql("SELECT * FROM gedi_dl.data").columns
```


In [3]:
# The query can also include computed columns.
# See https://duckdb.org/docs/stable/sql/functions/
# It is also possible to write your own functions in Python and register them with DuckDB,
# though you will find that many useful functions are already built-in.

# Here is an example of some of the most useful computed columns for GEDI data:
con.sql("""--sql
    SELECT
        -- Use a sql-compatible timestamp type instead of seconds since 2018-01-01
        TIMESTAMPTZ '2018-01-01T00:00:00Z' + (delta_time_l2a * INTERVAL '1 second') AS absolute_time,
        -- Extract the unique granule identifier (useful for computing variance)
        REGEXP_EXTRACT(root_file_l2a, '(O\\d{5}_\\d{2})') AS granule,
        -- Use a spatial function to convert lat/lon to a geometry type
        -- For more on spatial queries, see the 'Spatial queries' section, below.
        ST_Point(lat_lowestmode_l2a, lon_lowestmode_l2a) AS geom_4326,
    FROM gedi_dl.data
""")

┌───────────────────────────────┬───────────┬────────────────────────────────────────────────┐
│         absolute_time         │  granule  │                   geom_4326                    │
│   timestamp with time zone    │  varchar  │                    geometry                    │
├───────────────────────────────┼───────────┼────────────────────────────────────────────────┤
│ 2020-12-15 04:50:15.033526-05 │ O11379_01 │ POINT (-10.073391610770075 -68.12474112447354) │
│ 2020-12-15 04:50:15.050054-05 │ O11379_01 │ POINT (-10.072558169541816 -68.12412728227996) │
│ 2020-12-15 04:50:15.058318-05 │ O11379_01 │ POINT (-10.072141165949608 -68.12382052188109) │
│ 2020-12-15 04:50:15.066582-05 │ O11379_01 │ POINT (-10.07172427485012 -68.12351362648947)  │
│ 2020-12-15 04:50:15.074846-05 │ O11379_01 │ POINT (-10.071307541194018 -68.12320656603487) │
│ 2020-12-15 04:50:15.08311-05  │ O11379_01 │ POINT (-10.070890568824575 -68.12289966643836) │
│ 2020-12-15 04:50:15.091374-05 │ O11379_01 │ POIN

### SQL: WHERE

In [4]:
# We can now add a third SQL clause: WHERE
# WHERE specifies a filter condition that rows must meet to be included in the output

# Modify the filter condition below and test the changes to the output
# For example, try a filter that compares two different columns.
# Add those columns to the SELECT statement (don't forget commas!) to preview the data
# and check that your filter is working as you expect

con.sql(f"""--sql
    SELECT shot_number, agbd_l4a, rh_098_l2a, l4_quality_flag_l4a
    FROM gedi_dl.data
    WHERE l4_quality_flag_l4a = 1 AND agbd_l4a > 25
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬───────────┬────────────┬─────────────────────┐
│    shot_number     │ agbd_l4a  │ rh_098_l2a │ l4_quality_flag_l4a │
│       uint64       │   float   │   float    │        uint8        │
├────────────────────┼───────────┼────────────┼─────────────────────┤
│  87711100300261219 │  34.36054 │      10.08 │                   1 │
│  87711100300261758 │ 28.125336 │       9.19 │                   1 │
│  87711100300261762 │ 31.413815 │       9.67 │                   1 │
│  87711100300261767 │  76.34197 │      14.72 │                   1 │
│  87711100300261768 │  74.74202 │      14.57 │                   1 │
│  87711100300261830 │ 31.203169 │       9.64 │                   1 │
│  87711100300261831 │ 49.042267 │      11.91 │                   1 │
│  87711100300261832 │ 26.417126 │       8.93 │                   1 │
│  87711100300261893 │ 32.477192 │       9.82 │                   1 │
│  87711100300261932 │  58.19987 │      12.92 │                   1 │
│          ·        

A few things to notice:
- This took a little longer than the select without a filter, because DuckDB had to keep searching until it found 20 rows that satisfied our conditions
- If you rerun the same cell again, though, it should load more quickly -- DuckDB automatically caches results

### Spatial queries

We can also run spatial queries on the data. A list of spatial functions supported by DuckDB can be found [here](https://duckdb.org/docs/stable/core_extensions/spatial/functions).

The explanation of these spatial functions in the docs is pretty sparse, but they work the same as in PostGIS, so you can consult [the PostGIS documentation](https://postgis.net/docs/manual-3.6/reference.html) to understand what each function does (with pictures!)

However, note that not all PostGIS functions are implemented in DuckDB.

We'll start by querying the database for a small area of interest.

In [5]:
# Start by creating an example shapefile for an AOI.
# We are using a simple bounding box, but you can use any shapefile here.
import geopandas as gpd
from shapely.geometry import box

region_gdf = gpd.GeoDataFrame(
    geometry=[
        box(-63.2, -4.2, -62.8, -3.8),  
        box(-62.5, -4.0, -62.0, -3.5)
    ], 
    crs="EPSG:4326")

In [6]:
# Use a sqlutils helper function to convert python geometries into DuckDB geometries.
region = sqlutils.gdf_to_duck(con, region_gdf, geometry_columns=["geometry"])
# Alternatively, if you have shapefile, you can read it directly with DuckDB using ST_Read:
# region = con.sql("SELECT * FROM ST_Read('/path/to/shapefile.shp')")

con.sql("SELECT * FROM region")

┌────────────────────────────────────────────────────────────────────────┐
│                                geometry                                │
│                                geometry                                │
├────────────────────────────────────────────────────────────────────────┤
│ POLYGON ((-4.2 -62.8, -3.8 -62.8, -3.8 -63.2, -4.2 -63.2, -4.2 -62.8)) │
│ POLYGON ((-4 -62, -3.5 -62, -3.5 -62.5, -4 -62.5, -4 -62))             │
└────────────────────────────────────────────────────────────────────────┘

In [7]:
# On large databases, we can help DuckDB find the relevant data
# by adding a spatial filter clause. This makes use of the built-in H3 spatial
# indexing in the GEDI H3 file structure.
# Note that this function operates on a GeoSeries, not a DuckDB table.
fast_filter = sqlutils.geoseries_to_filter(region_gdf.geometry, resolution=3)
print(f"Fast spatial filter: {fast_filter}")

Fast spatial filter: h3_03 = ANY(['838ae6fffffffff', '838ae4fffffffff', '838a19fffffffff', '838a1bfffffffff'])


In [8]:
# Now we run the spatial query: 
# get the average AGBD and RH 98 for high-quality shots within the AOI. 
con.sql(f"""--sql
        -- Create a temporary table called 'gedi' with the fast filter applied,
        --   and compute the geometry column
        WITH gedi AS (
            SELECT 
                *, 
                ST_POINT(lat_lowestmode_l2a, lon_lowestmode_l2a) AS geom_4326
            FROM 
                gedi_dl.data
            WHERE ({fast_filter})
        )
        -- Perform the query
        SELECT 
            COUNT(shot_number) AS n_shots,
            AVG(agbd_l4a), 
            AVG(rh_098_l2a), 
        FROM 
            gedi,
            region AS r
        WHERE 
            ST_Contains(r.geometry, gedi.geom_4326) 
            AND l4_quality_flag_l4a = 1 
        """)

# REMINDER: If your shapefile is in a different CRS,
#  be sure to reproject the GEDI geometry column to match,
# e.g. in the WITH clause above, compute another geometry column like this:
# ST_Transform(geom_4326, 'EPSG:4326', 'your-crs-here') AS geom_your_crs

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────────┬────────────────────┐
│ n_shots │   avg(agbd_l4a)   │  avg(rh_098_l2a)   │
│  int64  │      double       │       double       │
├─────────┼───────────────────┼────────────────────┤
│  297419 │ 122.0272184422955 │ 20.380079113361404 │
└─────────┴───────────────────┴────────────────────┘

### SQL: GROUP BY

In [9]:
# The above query aggregated all shots within both rows of the AOI, 
# but we can also group by region to get separate results 
# for each polygon in the shapefile.
# This introduces a new SQL clause: GROUP BY
con.sql(f"""--sql
        -- This part is the same as above
        WITH gedi AS (
            SELECT 
                *, 
                ST_POINT(lat_lowestmode_l2a, lon_lowestmode_l2a) AS geom_4326
            FROM 
                gedi_dl.data
            WHERE ({fast_filter})
        )
        -- Perform the query, now grouping by the regions
        SELECT 
            r.geometry,  -- Include the geometry of the region in the output
            COUNT(shot_number) AS n_shots,
            AVG(agbd_l4a), 
            AVG(rh_098_l2a), 
        FROM gedi, region AS r 
        WHERE 
            ST_Contains(r.geometry, gedi.geom_4326)
            AND l4_quality_flag_l4a = 1
        GROUP BY r.geometry -- Add a GROUP BY clause on the regions
        """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────────────────────────────────────┬─────────┬────────────────────┬────────────────────┐
│                                geometry                                │ n_shots │   avg(agbd_l4a)    │  avg(rh_098_l2a)   │
│                                geometry                                │  int64  │       double       │       double       │
├────────────────────────────────────────────────────────────────────────┼─────────┼────────────────────┼────────────────────┤
│ POLYGON ((-4 -62, -3.5 -62, -3.5 -62.5, -4 -62.5, -4 -62))             │  190785 │ 120.99928226073659 │ 20.501261734550383 │
│ POLYGON ((-4.2 -62.8, -3.8 -62.8, -3.8 -63.2, -4.2 -63.2, -4.2 -62.8)) │  106634 │  123.8663579700139 │ 20.163264341491832 │
└────────────────────────────────────────────────────────────────────────┴─────────┴────────────────────┴────────────────────┘

In [ ]:
# Another use of GROUP BY (and spatial functions) is to compute statistics
# for 1 km x 1 km grid cells, as is done in the GEDI L4B product.
# Here we will compute mean AGBD for each 1 km cell over an AOI.

con.sql(f"""--sql
        -- Create a temporary table, 'gedi',
        -- with the columns we need and the fast spatial filter applied.
        WITH gedi AS (
            SELECT 
                *,
                -- Create a geometry column from the GEDI data 
                ST_POINT(lat_lowestmode_l2a, lon_lowestmode_l2a) AS geom_4326,
                -- Convert to EPSG:6933 (EASE-Grid)
                ST_Transform(geom_4326, 'EPSG:4326', 'EPSG:6933') AS geom_6933,
                -- Compute 1 km grid cell indices
                FLOOR(ST_X(geom_6933) / 1000) AS x_km,
                FLOOR(ST_Y(geom_6933) / 1000) AS y_km,
                -- Extract the unique granule identifier
                REGEXP_EXTRACT(root_file_l2a, '(O\\d\\d\\d\\d\\d_\\d\\d)') AS granule
            FROM 
                gedi_dl.data
            WHERE ({fast_filter})
        )
        -- Group by the grid cell indices to compute statistics for each cell
        SELECT
            x_km,
            y_km,
            COUNT(DISTINCT granule) AS n_granules,
            COUNT(shot_number) AS n_shots,
            AVG(agbd_l4a) AS mean_agbd
        FROM gedi
        WHERE l4_quality_flag_l4a = 1
        GROUP BY x_km, y_km
        -- HAVING is the GROUP BY equivalent of WHERE: 
        -- it filters groups instead of rows
        HAVING
            COUNT(shot_number) > 5
            AND COUNT(DISTINCT granule) > 2
        """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────┬────────────┬─────────┬────────────────────┐
│  x_km   │  y_km  │ n_granules │ n_shots │     mean_agbd      │
│ double  │ double │   int64    │  int64  │       double       │
├─────────┼────────┼────────────┼─────────┼────────────────────┤
│ -5984.0 │ -535.0 │          5 │      46 │ 21.713878263111997 │
│ -5983.0 │ -534.0 │          4 │      24 │ 50.570270153693855 │
│ -5981.0 │ -530.0 │          4 │      43 │ 33.880855572085046 │
│ -5974.0 │ -516.0 │          5 │      94 │   81.0459631507067 │
│ -5964.0 │ -499.0 │          5 │      90 │ 143.81485316885843 │
│ -5950.0 │ -471.0 │          3 │      37 │  67.92373321024147 │
│ -5935.0 │ -444.0 │          4 │      56 │ 19.999850319076877 │
│ -5952.0 │ -536.0 │         10 │     127 │  95.68578960636894 │
│ -5937.0 │ -563.0 │          6 │     100 │  130.6889523792267 │
│ -5976.0 │ -527.0 │          4 │      83 │ 102.29502403305237 │
│    ·    │    ·   │          · │       · │          ·         │
│    ·    │    ·   │     

### Converting back to pandas/geopandas

If at any point you are struggling to use DuckDB to do what you want, it's easy to convert back to pandas/geopandas.

For most column types, this is a no-copy operation, so it should be about as fast as reading with pandas.
It is often much faster than reading the raw data files with pandas, though,
 because DuckDB can automatically parallelize the read, loading multiple subfiles matching the query simultaneously

In [19]:
pdf = con.sql(f"""--sql
    SELECT 
        shot_number, 
        agbd_l4a, 
        pai_z_000_l2b, 
        pai_z_001_l2b, 
        wsci_l4c, 
        lat_lowestmode_l2a, 
        lon_lowestmode_l2a, 
    FROM gedi_dl.data
    WHERE 
        h3_03 = '838a1bfffffffff' 
        AND l4_quality_flag_l4a = 1 
        AND l2b_quality_flag_l2b = 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Performance considerations

The above queries were all restricted to relatively small amounts of data so that they could be run interactively. However, DuckDB is designed to handle long-running SQL queries on larger-than-memory datasets. An exploration of these use cases is in `tutorial_duckdb_performance.ipynb`